[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/05-matplotlib.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/05-matplotlib.ipynb)

# DS-MLOps Python Foundations
## Part 5: Matplotlib and Its Ecosystem

**Python 3.12+ | Author: Anthony Faustine**

### Before you begin

This notebook assumes you have completed Parts 1-4 (`01-python-core.ipynb`, `02-control-flow.ipynb`, `03-python-patterns.ipynb`, `04-numpy.ipynb`). If you have not, start there.

Part 5 covers the two layers of Python's most established plotting stack: **matplotlib**, the library almost everything else in the ecosystem is built on, and **seaborn**, which sits on top of it for statistical graphics. Both work on the same **university analytics platform** scenario from earlier parts, extended here to student exam results across several courses and two semesters, since a believable visualisation needs more than one number to plot.

Part 6 (`06-lets-plot.ipynb`) introduces a different way of thinking about plots entirely: the grammar of graphics. Part 7 (`07-data-storytelling.ipynb`) covers what makes a chart good and applies this project's own house style to both libraries.

| Topic | Why it matters |
|---|---|
| **Figure and Axes** | The object model every matplotlib call eventually goes through |
| **Core chart types** | Line, scatter, bar, histogram: the four you will use constantly |
| **Multiple Axes** | Comparing several views of the same data side by side |
| **Saving figures** | Resolution and format choices that matter for reports and papers |
| **Seaborn** | One line of statistical graphics, still a matplotlib `Axes` underneath |


### Callout guide

<table style='border-collapse:collapse;width:60%'>
<tr><td style='padding:6px 12px;background:#EAF3FA;border-left:4px solid #0369A1'><i class="bi bi-info-circle-fill" style="color:#0369A1"></i> <b>Key Concept</b></td><td style='padding:6px 12px'>Core idea with precise definition</td></tr>
<tr><td style='padding:6px 12px;background:#EAF7F0;border-left:4px solid #059669'><i class="bi bi-journal-code" style="color:#059669"></i> <b>Example</b></td><td style='padding:6px 12px'>Worked illustration</td></tr>
<tr><td style='padding:6px 12px;background:#FFF1E6;border-left:4px solid #EA580C'><i class="bi bi-puzzle-fill" style="color:#EA580C"></i> <b>Activity</b></td><td style='padding:6px 12px'>Hands-on challenge to try yourself</td></tr>
<tr><td style='padding:6px 12px;background:#FEF2F2;border-left:4px solid #DC2626'><i class="bi bi-bug-fill" style="color:#DC2626"></i> <b>Common Mistake</b></td><td style='padding:6px 12px'>Bug pattern to watch for</td></tr>
<tr><td style='padding:6px 12px;background:#F4F5F6;border-left:4px solid #6B7280'><i class="bi bi-lightbulb-fill" style="color:#6B7280"></i> <b>Pro Tip</b></td><td style='padding:6px 12px'>Production-grade habit</td></tr>
</table>

## Learning Objectives

By the end of Part 5 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Explain the Figure/Axes object model and why it matters | Sec. 1 |
| 2 | Build line, scatter, bar, and histogram charts with the object-oriented API | Sec. 2 |
| 3 | Lay out and compare multiple Axes in one Figure | Sec. 3 |
| 4 | Save a figure at the right resolution and format for its destination | Sec. 3 |
| 5 | Use seaborn for one-line statistical graphics, then keep customising with matplotlib | Sec. 4 |

## 1. Why Visualise? The Figure and Axes

A table of a thousand exam scores tells you nothing at a glance. A histogram of the same thousand scores tells you the shape of the distribution in about half a second. That is the entire case for visualisation: it trades a small amount of precision for a large amount of immediate understanding, which is exactly what you want before you have decided which question to ask next.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
exam_scores = rng.normal(loc=72, scale=12, size=1000).clip(0, 100)

print(f"mean   : {exam_scores.mean():.1f}")
print(f"median : {np.median(exam_scores):.1f}")
print(f"std    : {exam_scores.std():.1f}")
# The numbers alone do not tell you whether the distribution is symmetric,
# has a long tail, or is bimodal. A histogram answers that in one look.

matplotlib has two APIs for building the same chart. The older one, `pyplot` (`plt.plot(...)`), is a state machine: it always draws onto "whichever figure was most recently touched," which is fine for a single quick chart and confusing the moment you need two charts side by side. The **object-oriented API** is explicit instead: you ask for a `Figure` (the whole canvas) and one or more `Axes` (an individual plot inside it), then call methods directly on the `Axes` you want to draw on.

In [ ]:
import matplotlib.pyplot as plt

# The object-oriented pattern you will use for almost every chart in this
# notebook: ask for a Figure and an Axes, then call methods on the Axes.
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(exam_scores, bins=20, color="#4477AA", edgecolor="white")
ax.set_xlabel("Exam score")
ax.set_ylabel("Number of students")
ax.set_title("Exam score distribution");

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Figure vs. Axes</span><br><br>
A <code>Figure</code> is the whole canvas: the window or page a chart is drawn on, and the thing you save to a file. An <code>Axes</code> is one plot inside that canvas, with its own x-axis, y-axis, title, and data. <code>fig, ax = plt.subplots()</code> gives you one of each. Every method that actually draws data (<code>.plot()</code>, <code>.scatter()</code>, <code>.bar()</code>, <code>.hist()</code>) lives on the <code>Axes</code>, not the <code>Figure</code>.
</div>

The dashed boundary below is the one thing in this diagram that has no real line of code behind it: it is just there to make the Figure's own edge visible, since otherwise it is easy to forget it exists at all.

In [ ]:
from ark.plot.diagrams import figure_axes_diagram

figure_axes_diagram();

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Mixing <code>plt.plot()</code> and <code>ax.plot()</code></span><br><br>
<code>plt.title("x")</code> sets the title of whichever Axes <code>pyplot</code> thinks is "current", which silently changes after you create a new subplot. The moment you have more than one Axes, calling <code>plt.xlabel()</code> instead of <code>ax.set_xlabel()</code> is a common way to label the wrong chart. Once you have an <code>ax</code> object, call methods on it directly and skip <code>plt.*</code> entirely.
</div>

## 2. Core Chart Types

Four chart types cover most of what you will plot day to day. Each answers a different question, and picking the wrong one is the fastest way to make a chart that looks fine but says nothing useful.

In [ ]:
# Build the running dataset: exam results across three courses and two
# semesters for the university analytics platform.
rng = np.random.default_rng(42)

courses = np.array(["Machine Learning", "Data Structures", "Statistics"])
semesters = np.array(["Fall 2024", "Spring 2025"])

n_per_group = 60
course_col = np.repeat(courses, n_per_group * len(semesters))
semester_col = np.tile(np.repeat(semesters, n_per_group), len(courses))

# Each course has a slightly different difficulty and improves slightly
# from Fall to Spring, to give the line chart in this section something
# real to show.
course_base = {"Machine Learning": 68, "Data Structures": 74, "Statistics": 71}
semester_bump = {"Fall 2024": 0, "Spring 2025": 4}

exam_score = np.array(
    [rng.normal(course_base[c] + semester_bump[s], 10) for c, s in zip(course_col, semester_col, strict=True)]
).clip(0, 100)
study_hours = rng.uniform(0, 25, size=len(course_col))
attendance_pct = rng.uniform(50, 100, size=len(course_col))

print(f"rows: {len(course_col)}")
print(f"courses: {courses}")

**Line chart**, for a trend across an ordered sequence. Here, average score per course from Fall to Spring:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

for course in courses:
    course_mask = course_col == course
    means = [exam_score[course_mask & (semester_col == s)].mean() for s in semesters]
    ax.plot(semesters, means, marker="o", label=course)

ax.set_ylabel("Average exam score")
ax.set_title("Average score by course, Fall to Spring")
ax.legend();

**Scatter plot**, for the relationship between two continuous variables. Here, study hours against exam score:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(study_hours, exam_score, alpha=0.4, color="#4477AA")
ax.set_xlabel("Study hours")
ax.set_ylabel("Exam score")
ax.set_title("Study hours vs. exam score");

**Bar chart**, for comparing a single number across categories. Here, average score per course:

In [ ]:
course_means = [exam_score[course_col == c].mean() for c in courses]

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(courses, course_means, color=["#4477AA", "#EE6677", "#228833"])
ax.set_ylabel("Average exam score")
ax.set_title("Average score by course")
ax.tick_params(axis="x", labelrotation=15);

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Attendance Histogram</span><br><br>

<b>Goal:</b> Plot a histogram of <code>attendance_pct</code> with 15 bins, label both axes, and give it a title. Use the object-oriented pattern: <code>fig, ax = plt.subplots()</code>, then call methods on <code>ax</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(attendance_pct, bins=15, ...)
# expect a roughly uniform spread between 50 and 100, since
# attendance_pct was generated with rng.uniform(50, 100, ...)</pre>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
# TODO: plot the histogram, then set xlabel, ylabel, and title
...

fig

<div style='background:#F4F5F6;border-left:5px solid #6B7280;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#6B7280;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: A trailing bare <code>fig</code> displays the figure in Jupyter</span><br><br>
Jupyter displays the last expression in a cell automatically, the same way it prints a bare <code>x</code> on its own line. Ending a plotting cell with <code>fig</code> (or letting <code>ax.hist(...)</code> be the last call) shows the chart without an explicit <code>plt.show()</code>, which you only need outside a notebook.
</div>

## 3. Multiple Axes and Saving Figures

Real analysis rarely stops at one chart. `plt.subplots(nrows, ncols)` returns a whole grid of `Axes` at once, as a NumPy array, so you can loop over it the same way you looped over any other array in Part 4.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)

for ax, course in zip(axes.flat, courses, strict=True):
    course_scores = exam_score[course_col == course]
    ax.hist(course_scores, bins=15, color="#4477AA", edgecolor="white")
    ax.set_title(course, fontsize=10)
    ax.set_xlabel("Exam score")

axes[0].set_ylabel("Number of students")
fig.suptitle("Score distribution per course");

`axes.flat` works regardless of the grid shape: a `2x2` grid of `Axes` is a 2D array, but `.flat` always gives you a flat iterator over all of them, in row-major order. `sharey=True` forces every Axes in the grid to use the same y-axis range, which is what makes the three histograms above honestly comparable instead of each rescaled to its own data.

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Comparing histograms with different y-axis scales</span><br><br>
Without <code>sharey=True</code>, matplotlib autoscales each Axes to its own data. Three histograms that look like they have the same number of students can actually have wildly different counts, because each y-axis silently uses a different scale. Whenever you put similar charts side by side for comparison, force a shared scale.
</div>

Saving a figure has two choices that matter: resolution (`dpi`, dots per inch) and file format. A raster format (PNG) at low `dpi` looks blurry when scaled up; a vector format (SVG or PDF) stays sharp at any size because it stores shapes, not pixels:

In [ ]:
from pathlib import Path

output_dir = Path("tmp_plots")
output_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(exam_scores, bins=20, color="#4477AA", edgecolor="white")
ax.set_title("Exam score distribution")

# PNG: fine for slides and notebooks, blurry if you zoom in or print large
fig.savefig(output_dir / "scores.png", dpi=150)
# SVG: vector format, stays crisp at any size, the right choice for reports
fig.savefig(output_dir / "scores.svg")

print(sorted(p.name for p in output_dir.iterdir()))

import shutil

shutil.rmtree(output_dir)

<div style='background:#F4F5F6;border-left:5px solid #6B7280;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#6B7280;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Default to vector formats for anything that gets printed or zoomed</span><br><br>
PNG and JPEG store a fixed grid of pixels: stretch them and they blur. SVG and PDF store the actual shapes and text, so they render sharp at any zoom level or print size. Save PNG for quick previews and web thumbnails; save SVG or PDF for anything that ends up in a report, slide deck, or paper.
</div>

## 4. Seaborn: Statistical Graphics in One Line

Seaborn is built directly on matplotlib. It does not replace anything from Sections 1-3: it adds a layer of functions that know how to take a whole DataFrame, split it by a category, color each group, and draw a legend, all in a single call. Reaching for seaborn first whenever your chart needs grouping or a statistical summary saves a genuine amount of code.

Seaborn expects **tidy data**: one row per observation, one column per variable. A full pandas introduction comes later in the data analysis tutorials, but building a DataFrame from arrays you already have is one line:

In [ ]:
import pandas as pd

results = pd.DataFrame(
    {
        "course": course_col,
        "semester": semester_col,
        "exam_score": exam_score,
        "study_hours": study_hours,
        "attendance_pct": attendance_pct,
    }
)
results.head()

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.histplot(data=results, x="exam_score", hue="course", kde=True, ax=ax)
ax.set_title("Exam score distribution by course");

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Seaborn returns a matplotlib Axes</span><br><br>
<code>sns.histplot(..., ax=ax)</code> draws onto the <code>Axes</code> you pass it and returns that same <code>Axes</code>. Nothing from Sections 1-3 is wasted: every <code>ax.set_title()</code>, <code>ax.set_xlabel()</code>, or <code>fig.savefig()</code> you already know still works on a seaborn chart. Seaborn only replaces the part where you would otherwise have looped over groups and called <code>ax.hist()</code> once per group yourself.
</div>

`sns.boxplot` summarises a whole distribution (median, quartiles, outliers) per category in one call, the kind of comparison that would take a loop and several `ax.hist()` calls in raw matplotlib:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=results, x="course", y="exam_score", hue="semester", ax=ax)
ax.set_title("Exam score by course and semester");

`sns.heatmap` is the standard way to visualise a correlation matrix. Pass it `results[numeric_cols].corr()`, a small DataFrame seaborn happily turns into a colour grid with the actual numbers annotated:

In [ ]:
numeric_cols = ["exam_score", "study_hours", "attendance_pct"]
corr = results[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature correlation");

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Compare Study Habits Across Courses</span><br><br>

<b>Goal:</b> Use <code>sns.boxplot</code> to compare <code>study_hours</code> across the three courses (x-axis: <code>course</code>, y-axis: <code>study_hours</code>), then add a title with <code>ax.set_title()</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=results, x="course", y="study_hours", ax=ax)
ax.set_title(...)</pre>
<b>Hint:</b> This is almost identical to the boxplot above, just with a different y-axis and no <code>hue</code>.
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
# TODO: boxplot of study_hours by course, plus a title
...

fig

<div style='background:#F4F5F6;border-left:5px solid #6B7280;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#6B7280;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Seaborn's defaults are a starting point, not a destination</span><br><br>
<code>sns.histplot</code>, <code>sns.boxplot</code>, and friends choose sensible colours and spacing automatically so you can explore data fast. Part 7 covers replacing those defaults with this project's own house style once you are past exploring and ready to present a chart to someone else.
</div>

## Capstone: A Three-Panel Course Report

Combine everything from this notebook into one Figure with three Axes side by side: a histogram of scores for one course, a scatter of study hours against score for the same course, and a bar chart comparing average scores across all three courses. This is the shape of report you would actually hand to a department head.

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Three-Panel Report</span><br><br>

<b>Goal:</b> Build a <code>(1, 3)</code> grid of Axes:
<ol>
<li>Axes 0: histogram of <code>exam_score</code> for <code>"Machine Learning"</code> only</li>
<li>Axes 1: scatter of <code>study_hours</code> vs. <code>exam_score</code>, same course only</li>
<li>Axes 2: bar chart of average <code>exam_score</code> per course (all courses)</li>
</ol>
Give the Figure an overall title with <code>fig.suptitle()</code> and each Axes its own <code>ax.set_title()</code>. <b>Hint:</b> Filter with <code>ml_mask = results["course"] == "Machine Learning"</code>, then index <code>results[ml_mask]</code> for the first two panels.
</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

ml_mask = results["course"] == "Machine Learning"
ml_results = results[ml_mask]

# TODO panel 0: histogram of ml_results["exam_score"]
# TODO panel 1: scatter of ml_results["study_hours"] vs ml_results["exam_score"]
# TODO panel 2: bar chart of average exam_score per course (all courses)
...

fig.suptitle("Course Report: Machine Learning")
fig

## Summary

| Concept | Key rule |
|---|---|
| Figure vs. Axes | Figure is the canvas, Axes is one plot; draw by calling methods on `ax`, not `plt` |
| `plt.subplots()` | Returns `(fig, ax)` for one plot, `(fig, axes)` for a grid |
| Line chart | Trend across an ordered sequence: `ax.plot()` |
| Scatter plot | Relationship between two continuous variables: `ax.scatter()` |
| Bar chart | Comparing one number across categories: `ax.bar()` |
| Histogram | Shape of one variable's distribution: `ax.hist()` |
| `axes.flat` | Flat iterator over any subplot grid, regardless of its shape |
| `sharey=True` | Forces a fair visual comparison across subplots |
| Saving figures | PNG for previews, SVG/PDF for anything printed or zoomed |
| Seaborn | One-line statistical graphics on tidy DataFrames; returns a matplotlib `Axes` |

**Next:** `06-lets-plot.ipynb`, introducing the grammar of graphics: the same charts from this notebook, built declaratively instead of imperatively.